# Blackjack with Dynamic Programming (Value Iteration)

## Goal
I want to find the best possible Blackjack strategy using **Dynamic Programming**. Dynamic Programming is model based, which means I need to know the transition probabilities of the game before I start. Gymnasium's `Blackjack-v1` does not give me an `env.P` like FrozenLake did, so I build that model myself.

## The one assumption I make
Cards are drawn from an **infinite deck with replacement**. Every draw is independent, and the odds never change no matter which cards came out earlier. This is exactly how Gymnasium's Blackjack works, so my hand built model matches the real environment. It also keeps my state a valid Markov state, because I do not need to remember which cards are already gone.

## My state
`(player_sum, dealer_up_card, ace_as_eleven)`

* `player_sum`: the current total of my hand
* `dealer_up_card`: the one dealer card I can see (1 means an ace, 10 covers 10, J, Q, K)
* `ace_as_eleven`: whether my hand has an ace that is currently counted as 11 (a "soft" hand)

## My actions
Stick (stop and let the dealer play) or Hit (take one more card).

## Rewards
Win: +1. Lose: -1. Draw: 0. I use `gamma = 1.0` because one hand of Blackjack is short and I do not want to discount anything inside it.

## The plan
1. Card probabilities: the fixed odds per rank, the building block for everything else.
2. Dealer outcome distribution: for each dealer up card (1 to 10), the probability that the dealer ends on 17, 18, 19, 20, 21 or busts.
3. `stick_value` and `hit_value`: the expected return of each action, built from steps 1 and 2.
4. Value Iteration: the same mechanical shape as FrozenLake, just fed with my own derived model.
5. Testing: extract the policy and play real hands with `gym.make("Blackjack-v1")`.

In [1]:
import numpy as np
import gymnasium as gym

## Step 1: Card probabilities

There are 13 ranks in a deck, and each rank has the same chance of being drawn. Ten, Jack, Queen and King all count as 10, so the value 10 is four ranks together: `4/13`. Every other value is `1/13`.

In [2]:
card_prob = {
    1: 1/13,    # ace (counts as 1 or 11, handled by the helper below)
    2: 1/13,
    3: 1/13,
    4: 1/13,
    5: 1/13,
    6: 1/13,
    7: 1/13,
    8: 1/13,
    9: 1/13,
    10: 4/13    # covers 10, J, Q, K together
}

# sanity checks: 10 distinct card values, and the probabilities must sum to 1
print(len(card_prob))
print(sum(card_prob.values()))

10
1.0


## Step 2a: The ace helper

An ace is special because it can count as 1 or as 11, whichever is better for the hand. I track this with one flag.

**`ace_as_eleven` means: there is an ace in this hand that is currently being counted as 11.**

The rule my helper follows when a card arrives:

1. Add the card at its face value first. An ace counts as 1 at this stage.
2. If the new card is an ace, I do not already have an ace counted as 11, and adding 10 more would not go over 21, then I promote this ace to 11.
3. If I am over 21 and I have an ace counted as 11, I demote it by subtracting 10.

**The bug I fixed here:** my older version added 11 for every ace. A second ace in a hand that already had one counted as 11 would also be counted as 11, which is impossible (two elevens make 22). Adding the ace as 1 first and promoting it only when it is safe removes that problem.

In [3]:
def ace_value(sum_cards, ace_as_eleven, new_card):
    card_sum = sum_cards + new_card              # every card, including an ace, adds its face value first (ace = 1)

    if new_card == 1 and not ace_as_eleven and card_sum + 10 <= 21:
        card_sum = card_sum + 10                 # promote this ace to 11, it is safe
        ace_as_eleven = True

    if ace_as_eleven and card_sum > 21:
        card_sum = card_sum - 10                 # over 21, so demote the ace I was counting as 11
        ace_as_eleven = False

    return card_sum, ace_as_eleven


print(ace_value(15, True, 1))      # expect (16, True): second ace counts as 1, the first stays 11
print(ace_value(21, True, 1))      # expect (12, False): 22 busts, so the first ace drops to 1
print(ace_value(5, False, 1))      # expect (16, True): 5 + 11 = 16 is safe, so the ace becomes 11
print(ace_value(15, False, 1))     # expect (16, False): 15 + 11 = 26 would bust, so the ace stays 1

(16, True)
(12, False)
(16, True)
(16, False)


## Step 2b: Dealer outcome distribution

The dealer has a fixed rule: **hit below 17, stand on 17 or more**. So the dealer never makes a decision, which means I can compute exactly how likely each final result is for any up card.

I do it with **recursion**. At each point the dealer either has already finished (17 or more) or must draw one more card, which branches into 10 possibilities.

The four inputs:

* `dealer_sum`: the dealer's total at this point in the recursion
* `ace_as_eleven`: whether the dealer has an ace counted as 11
* `path_probability`: the probability of this exact sequence of cards so far. Going one level deeper always **multiplies** by the next card's probability, because both events must happen together.
* `outcomes`: one shared dictionary passed into every call. It accumulates the final answer: one probability per possible dealer result.

In [4]:
def dealer_distribution(dealer_sum, ace_as_eleven, path_probability, outcomes):

    # base case: the dealer must stop at 17 or higher, so this branch ends here
    if dealer_sum >= 17:
        if dealer_sum > 21:
            outcomes["bust"] = outcomes["bust"] + path_probability
        else:
            outcomes[dealer_sum] = outcomes[dealer_sum] + path_probability
        return

    # the dealer must hit: branch into every possible next card, each weighted by its own probability
    for next_card in card_prob:
        card_probability = card_prob[next_card]      # this specific card's chance, for example 1/13
        new_sum, new_ace_as_eleven = ace_value(dealer_sum, ace_as_eleven, next_card)

        # path so far TIMES this card's probability
        dealer_distribution(new_sum, new_ace_as_eleven, path_probability * card_probability, outcomes)

### Building the table for every up card

`dealer_outcome_table[up_card]` holds the complete distribution over the dealer's final result for that up card.

Two details that matter:

* A fresh `outcomes` dictionary is created inside every loop, so no up card ever shares or reuses another one's numbers.
* `path_probability` starts at `1.0` because nothing has happened yet, so reaching the starting point is certain.
* An ace up card starts at **11** with `ace_as_eleven = True`, because a lone ace counts as 11. (Starting it at 1 was my second bug and made the ace bust rate 0.18 instead of the true 0.115.)

In [5]:
dealer_outcome_table = {}

for up_card in range(1, 11):                      # every possible up card: 1 (ace) through 10

    outcomes = {17: 0.0, 18: 0.0, 19: 0.0, 20: 0.0, 21: 0.0, "bust": 0.0}

    if up_card == 1:
        dealer_distribution(11, True, 1.0, outcomes)           # a lone ace counts as 11
    else:
        dealer_distribution(up_card, False, 1.0, outcomes)     # no ace involved yet

    dealer_outcome_table[up_card] = outcomes

### Checking the table

Every row must sum to 1, because the six outcomes together cover everything the dealer can end up with.

**Fix in this cell:** my old print used `{probability:.2f}%`. That only attaches a percent sign to a fraction, so 0.42 showed up as `0.42%` when it really means 42%. The format `:.2%` multiplies by 100 and adds the sign correctly.

In [6]:
# every row should sum to 1.0
for up_card in dealer_outcome_table:
    print(up_card, sum(dealer_outcome_table[up_card].values()))

print()

# the distribution for a dealer showing a 6: a weak card, so a high bust rate
for outcome, probability in dealer_outcome_table[6].items():
    print(f"{outcome}: {probability:.2%}")

1 0.9999999999999997
2 0.9999999999999957
3 0.9999999999999977
4 0.9999999999999991
5 0.9999999999999996
6 0.9999999999999998
7 0.9999999999999998
8 1.0
9 1.0000000000000002
10 1.0000000000000002

17: 16.54%
18: 10.63%
19: 10.63%
20: 10.17%
21: 9.72%
bust: 42.32%


## Step 3a: The value of sticking

If I stick, I make no further decisions. The dealer plays out, and my reward depends on where the dealer ends up compared with my `player_sum`:

* dealer busts: +1
* dealer ends higher than me: -1
* dealer ends lower than me: +1
* dealer ties me: 0

The value of sticking is the **probability weighted sum** of those rewards across every dealer outcome:

$$\text{stick value} = \sum_{\text{dealer result}} P(\text{dealer result}) \times \text{reward}$$

`elif` is used, not separate `if` statements, since only one case applies per outcome. A busted player always gets -1, so I guard that case first. (`player_sum > 21` is never a real input from the game, but without the guard a sum like 23 would wrongly count as beating every dealer total.)

In [7]:
def stick_value(player_sum, dealer_up_card):

    if player_sum > 21:                          # guard: a busted hand is always -1
        return -1.0

    total_rewards = 0.0

    # loop through this up card's own distribution only
    for dealer_result, probability in dealer_outcome_table[dealer_up_card].items():

        if dealer_result == "bust":
            reward = 1          # dealer busted, I win
        elif dealer_result > player_sum:
            reward = -1         # dealer ended higher, I lose
        elif dealer_result < player_sum:
            reward = 1          # dealer ended lower, I win
        else:
            reward = 0          # equal totals, a draw

        # weight this outcome's reward by how likely it is, then add it to the running total
        total_rewards = total_rewards + probability * reward

    return total_rewards


print(stick_value(18, 6))    # 18 against a weak 6: clearly positive
print(stick_value(21, 6))    # 21 only fails to win when the dealer also makes 21
print(stick_value(12, 10))   # 12 against a 10: clearly negative
print(stick_value(23, 6))    # busted: must be -1.0

0.2834439160468988
0.9028367438425797
-0.5757818467646018
-1.0


## Step 3b: The value function table `V`

`V` stores my current estimate of how good each state is. I need a lookup by state, so it is a **dictionary keyed by the state tuple**, not a numpy array.

Line by line:

1. `V = {}`: start with an empty dictionary.
2. `for player_sum in range(4, 22)`: every hand total I can be in while still playing. The lowest possible starting hand is 2 + 2 = 4, and 21 is the highest total before busting.
3. `for dealer_up_card in range(1, 11)`: all ten visible dealer cards (1 is the ace).
4. `for ace_as_eleven in (False, True)`: each combination exists with and without a soft ace.
5. `if ace_as_eleven and player_sum < 12: continue`: **my fix.** A hand with an ace counted as 11 cannot total less than 12 (the smallest is ace + ace = 12), so those states can never happen. Skipping them keeps `V` limited to real states.
6. `V[(player_sum, dealer_up_card, ace_as_eleven)] = 0.0`: the key is the state, the value starts at zero. Value Iteration will improve every one of these numbers.

In [8]:
V = {}

for player_sum in range(4, 22):
    for dealer_up_card in range(1, 11):
        for ace_as_eleven in (False, True):

            # a soft hand (ace counted as 11) cannot total below 12, so these states never occur
            if ace_as_eleven and player_sum < 12:
                continue

            V[(player_sum, dealer_up_card, ace_as_eleven)] = 0.0

print(len(V))   # 18 * 10 hard states + 10 * 10 soft states = 280

280


## Step 3c: The value of hitting

If I hit, I draw one card and land in a new state. I do not know which card I will get, so I take an **expectation over all 10 possible cards**:

$$\text{hit value} = \sum_{\text{card}} P(\text{card}) \times \big(\text{reward} + \gamma \cdot V(\text{next state})\big)$$

What each piece means:

* `P(card)`: the chance of drawing that card (`card_prob[next_card]`).
* `reward`: what I receive immediately for this draw. It is 0 while the hand is still alive, because nothing is won or lost until the hand ends.
* `gamma * V(next state)`: the long term value of the position I land in, assuming I play well from there. With `gamma = 1.0` it is simply `V(next state)`.
* **Bust case:** if the draw takes me over 21 the game ends right away with reward -1. There is no next state, so `V` is not used at all.

The loop computes that formula for every card and adds the pieces into `total`. The important habit: `total` is only ever added to, never reset inside the loop.

In [9]:
gamma = 1.0   # no discounting inside one hand of Blackjack


def hit_value(player_sum, dealer_up_card, ace_as_eleven, V):
    total = 0.0                                                          # expected value over all cards

    for next_card in card_prob:
        probability = card_prob[next_card]                               # chance of drawing this card
        new_sum, new_ace = ace_value(player_sum, ace_as_eleven, next_card)   # state after the draw

        if new_sum > 21:
            # bust: the game ends, reward -1, no next state, so V is NOT used
            total = total + probability * (-1)
        else:
            # still alive: reward 0 for this step, then continue from the new state
            reward = 0
            total = total + probability * (reward + gamma * V[(new_sum, dealer_up_card, new_ace)])

    return total

### Why do some of these tests print 0.0?

Nothing is wrong. There are two reasons that both have to hold for the answer to be exactly zero:

1. **No card can bust.** For `(12, 6, True)` the 12 is a soft hand. Even a 10 gives 22, but then the ace drops from 11 to 1 and the total becomes 12 again. For `(4, 6, False)` the worst case is 4 + 10 = 14. With no bust possible, the `-1` branch never runs.
2. **`V` is all zeros right now.** Every surviving card contributes `probability * (0 + 1.0 * 0)`, which is 0.

So the whole sum is 0.0. For `(12, 6, False)` only a 10 busts (12 + 10 = 22), which gives `4/13 * (-1) = -0.3077`, and the other cards still contribute 0. For `(20, 6, False)` only an ace survives, so 12 of the 13 ranks bust: `-12/13 = -0.923`.

Once Value Iteration fills `V` with real values, the zeros disappear. The last cell below proves this by loading `V` with a pretend value of 0.5 everywhere.

In [10]:
print(hit_value(20, 6, False, V))   # only an ace survives, so 12/13 busts: about -0.923
print(hit_value(12, 6, False, V))   # only a 10 busts: -4/13, about -0.3077
print(hit_value(12, 6, True, V))    # soft 12: no card can bust, and V is all zeros: 0.0
print(hit_value(4, 6, False, V))    # 4 + 10 = 14 at most: no bust, V is all zeros: 0.0

-0.9230769230769231
-0.3076923076923077
0.0
0.0


In [11]:
# proof that the zeros come from the empty V: fill a copy with 0.5 and ask again
V_demo = {}
for state in V:
    V_demo[state] = 0.5

print(hit_value(12, 6, True, V_demo))   # no bust possible, every next state is worth 0.5: expect 0.5
print(hit_value(4, 6, False, V_demo))   # expect 0.5 as well
print(hit_value(12, 6, False, V_demo))  # 9/13 survive at 0.5 each, 4/13 bust at -1: 0.5/13, about 0.038

0.5
0.5
0.038461538461538436


## Step 4: Value Iteration

Now I have everything I need: the value of sticking and the value of hitting in every state. Value Iteration repeats one simple idea until nothing changes:

$$V(s) = \max\big(\text{stick value}(s),\ \text{hit value}(s)\big)$$

How one **sweep** works:

1. Start a fresh dictionary `new_V = {}` and set `delta = 0.0`.
2. For every state, compute the stick value and the hit value, using the **old** `V` for every lookup.
3. Store the better of the two in `new_V[state]`.
4. Track `delta`, the biggest change in any state during this sweep.
5. After the sweep, `V = new_V`. If `delta` is below `theta`, the values have converged and I stop.

Two details that matter:

* `new_V` is a snapshot. Every state in one sweep is judged against the same old numbers, and `V` is only replaced after the whole pass.
* The stick value never depends on `V`, so only the hit side changes between sweeps.

It converges quickly because hitting only moves a hand forward (a higher total, or a soft ace turning hard). It never loops back, so each sweep fixes one more layer of states, starting from the high totals.

In [12]:
theta = 1e-6      # convergence threshold: stop when no state changes by more than this
sweeps = 0        # counts how many full passes over all states I do

while True:                                  # keep sweeping until the values stop changing
    delta = 0.0                              # biggest change in THIS sweep, reset every sweep
    new_V = {}                               # fresh snapshot every sweep: read old V, write new_V

    for state in V:
        player_sum, dealer_up_card, ace_as_eleven = state

        hit_q = hit_value(player_sum, dealer_up_card, ace_as_eleven, V)   # expected value if I draw a card
        stick_q = stick_value(player_sum, dealer_up_card)                 # expected value if I stop

        new_V[state] = max(hit_q, stick_q)   # the Bellman optimality step: take the better action

        change = abs(new_V[state] - V[state])
        delta = max(delta, change)           # keep the biggest change seen in this sweep

    V = new_V                                # the sweep is done: the snapshot becomes the current table
    sweeps = sweeps + 1

    if delta < theta:                        # even the biggest change is tiny, so V has converged
        break

print("sweeps:", sweeps)
print(V[(20, 6, False)])
print(V[(16, 10, False)])

sweeps: 9
0.7039585701713446
-0.5693071598807666


### Reading these numbers

* **`sweeps: 9`**: the values stopped changing after nine passes over all 280 states.
* **`V[(20, 6, False)]` is about 0.70**: holding a hard 20 against a dealer 6 and playing perfectly, I win about 0.70 chips per 1 chip bet on average.
* **`V[(16, 10, False)]` is about -0.57**: a hard 16 against a dealer 10 is the worst spot in the game. Even with perfect play I lose about 0.57 per chip on average. The best I can do is lose less.

A value of +1 would mean winning every time, and -1 would mean losing every time.

## Step 5: Extracting the policy

`V` tells me how good each state is. The **policy** tells me what to actually do in each state: compare the two actions using the final `V` and keep the better one.

Since the policy is built once from the converged `V`, there is no sweep loop this time. On an exact tie, both actions are equally good, so it does not matter which one I pick.

In [13]:
policy = {}

for state in V:
    player_sum, dealer_up_card, ace_as_eleven = state

    hit_q = hit_value(player_sum, dealer_up_card, ace_as_eleven, V)
    stick_q = stick_value(player_sum, dealer_up_card)

    if stick_q > hit_q:
        policy[state] = "stick"
    else:
        policy[state] = "hit"

# count how many states say stick
count = 0
for state in policy:
    if policy[state] == "stick":
        count = count + 1

print("states where I stick:", count, "out of", len(policy))

states where I stick: 110 out of 280


### The policy as a table

`H` means hit and `S` means stick. Rows are my hand total, columns are the dealer's visible card (1 is the ace). Hard hands have no ace counted as 11, soft hands do.

In [14]:
for ace_flag in (False, True):

    if ace_flag:
        print("SOFT hands (an ace counted as 11)")
    else:
        print("HARD hands")

    header = "sum | "
    for dealer_card in range(1, 11):
        header = header + f"{dealer_card:>2} "
    print(header)
    print("-" * len(header))

    for player_sum in range(4, 22):
        if ace_flag and player_sum < 12:
            continue                           # soft hands below 12 do not exist

        row = f"{player_sum:>3} | "
        for dealer_card in range(1, 11):
            if policy[(player_sum, dealer_card, ace_flag)] == "hit":
                letter = "H"
            else:
                letter = "S"
            row = row + f"{letter:>2} "
        print(row)
    print()

HARD hands
sum |  1  2  3  4  5  6  7  8  9 10 
------------------------------------
  4 |  H  H  H  H  H  H  H  H  H  H 
  5 |  H  H  H  H  H  H  H  H  H  H 
  6 |  H  H  H  H  H  H  H  H  H  H 
  7 |  H  H  H  H  H  H  H  H  H  H 
  8 |  H  H  H  H  H  H  H  H  H  H 
  9 |  H  H  H  H  H  H  H  H  H  H 
 10 |  H  H  H  H  H  H  H  H  H  H 
 11 |  H  H  H  H  H  H  H  H  H  H 
 12 |  H  H  H  S  S  S  H  H  H  H 
 13 |  H  S  S  S  S  S  H  H  H  H 
 14 |  H  S  S  S  S  S  H  H  H  H 
 15 |  H  S  S  S  S  S  H  H  H  H 
 16 |  H  S  S  S  S  S  H  H  H  H 
 17 |  S  S  S  S  S  S  S  S  S  S 
 18 |  S  S  S  S  S  S  S  S  S  S 
 19 |  S  S  S  S  S  S  S  S  S  S 
 20 |  S  S  S  S  S  S  S  S  S  S 
 21 |  S  S  S  S  S  S  S  S  S  S 

SOFT hands (an ace counted as 11)
sum |  1  2  3  4  5  6  7  8  9 10 
------------------------------------
 12 |  H  H  H  H  H  H  H  H  H  H 
 13 |  H  H  H  H  H  H  H  H  H  H 
 14 |  H  H  H  H  H  H  H  H  H  H 
 15 |  H  H  H  H  H  H  H  H

### Does this look like real Blackjack strategy?

Yes. Some well known rules show up in the table without me ever programming them:

* Always hit on low totals, where busting is impossible or very unlikely.
* A hard 13 to 16 against a weak dealer card (2 to 6) is a stick, and a hard 12 is a stick against a 4, 5 or 6: I let the dealer bust.
* A hard 12 to 16 against a strong dealer card (7 to ace) is a hit: sticking would mostly lose.
* A hard 17 or more is a stick.
* Soft hands are played more boldly: I keep hitting up to a soft 17, because the ace can drop from 11 to 1 and protect me from busting. A soft 18 sticks against most cards but hits against a 9, 10 or ace.

## Step 6: Testing the policy in the real game

Everything so far used my hand built model. Now I test the policy in Gymnasium's real `Blackjack-v1`, which the policy has never seen.

For every hand: reset the game, look up the action in my policy, convert the word into the number the environment expects (**0 is stick, 1 is hit**), and step until the hand ends. The last reward is +1 for a win, 0 for a draw and -1 for a loss.

To judge the result, I also play 100,000 hands with two simple baseline players: one who picks randomly, and one who always sticks. The outer loop picks which player is being tested. The only place where they differ is how the action is chosen.

In [15]:
episodes = 100000
env = gym.make("Blackjack-v1")      # built once, reused for all three players

# seeds, so the numbers can be reproduced
env.reset(seed=0)
env.action_space.seed(0)

results = {}

for strategy in ["policy", "random", "stick"]:

    # counters are reset for EACH player, so the results never mix
    wins = 0
    draws = 0
    losses = 0
    total_reward = 0.0

    for episode in range(episodes):

        terminated = False
        truncated = False

        # curr_state is (player_sum, dealer_card, usable_ace)
        curr_state, extra = env.reset()

        while not (terminated or truncated):

            if strategy == "policy":
                if policy[curr_state] == "hit":      # my Value Iteration strategy
                    curr_action = 1                  # 1 = hit
                else:
                    curr_action = 0                  # 0 = stick
            elif strategy == "random":
                curr_action = env.action_space.sample()   # coin flip every decision
            else:
                curr_action = 0                      # "stick": always stop

            next_state, reward, terminated, truncated, extra = env.step(curr_action)
            curr_state = next_state

        # the hand is over: the last reward is the result
        total_reward = total_reward + reward

        if reward > 0:
            wins = wins + 1
        elif reward < 0:
            losses = losses + 1
        else:
            draws = draws + 1

    print("strategy:", strategy)
    print(f"win rate:       {wins / episodes:.2%}")
    print(f"draw rate:      {draws / episodes:.2%}")
    print(f"loss rate:      {losses / episodes:.2%}")
    print(f"average reward: {total_reward / episodes:.4f}")
    print()

strategy: policy
win rate:       43.26%
draw rate:      9.01%
loss rate:      47.73%
average reward: -0.0447

strategy: random
win rate:       28.16%
draw rate:      4.26%
loss rate:      67.58%
average reward: -0.3942

strategy: stick
win rate:       38.59%
draw rate:      4.89%
loss rate:      56.52%
average reward: -0.1793



### What the results mean

* My policy wins about 43% of hands and loses about 47.5%, with about 9% draws. That is a small loss on average (around -0.04 per hand), and it is the **best result any strategy can reach in this game**.
* Blackjack always favors the house. I act first, so if I bust I lose immediately, even when the dealer would have busted right after. The dealer has no decisions to get wrong.
* The fair way to judge it is against the baselines: the random player loses about 0.4 per hand and always sticking loses about 0.18. My policy loses roughly 4 to 9 times less.
* The gap between wins and losses (about 4 percentage points) equals the average reward, because every win pays +1 and every loss costs 1.
* Real casinos allow doubling down and splitting pairs, which this simplified environment does not have. That is why real basic strategy gets the house edge close to 0.5%, while here about 4% is the ceiling.

## Conclusion

I built the full transition model by hand from card probabilities, solved the game with Value Iteration in 9 sweeps, extracted the optimal policy, and confirmed in the real environment that it beats both baselines by a wide margin. The model, derived without ever looking at the environment's internals, plays as well as the best known strategy for this version of the game.